# Benchmark 4DVar à critère de convergence commun

Ce notebook compare **SciPy historique** et **scan + Optax découplé** jusqu’à la même condition de stationnarité : `||gₖ||₂ / ||g₀||₂ ≤ 0.10` pendant deux itérations acceptées consécutives. Le test commence après au moins cinq itérations et s’arrête au plus tard après 50 itérations.

In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASELINE_REF = "1f75752"
GPU = "1"
WINDOW_DAYS = 0
MAX_ITERATIONS = 50
HISTORY_SIZE = 10
RELATIVE_GRADIENT_TOLERANCE = 0.10
CONVERGENCE_PATIENCE = 2
MINIMUM_ITERATIONS = 5


## Protocole

Les deux variantes utilisent le même contrôle nul, les mêmes observations, bases, conditions aux limites, mémoire L-BFGS et critère d’arrêt. Elles sont exécutées dans deux processus froids indépendants sur le même GPU. Les temps de setup, compilation et minimisation restent séparés.

In [2]:
def command(*args, cwd=None, check=True):
    return subprocess.run([str(arg) for arg in args], cwd=cwd, check=check, text=True, capture_output=True)

repo_root = Path(command("git", "rev-parse", "--show-toplevel").stdout.strip()).resolve()
whirls_dir = repo_root / "mapping" / "examples" / "WHIRLS"
config_path = whirls_dir / "config_VarDyn-QG.py"
driver_path = whirls_dir / "benchmark_4dvar_minimizers.py"
results_path = whirls_dir / "benchmark_4dvar_convergence_results.json"
baseline_sha = command("git", "rev-parse", BASELINE_REF, cwd=repo_root).stdout.strip()

def run_variant(label, optimizer, source_root, cache_dir, resident, jit, schedule):
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": GPU,
        "MPLBACKEND": "Agg",
        "PYTHONPATH": str(source_root / "mapping"),
        "PYTHONUNBUFFERED": "1",
        "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
        "JAX_COMPILATION_CACHE_DIR": str(cache_dir),
    })
    process = subprocess.run([
        sys.executable, str(driver_path),
        "--config", str(config_path),
        "--label", label,
        "--optimizer", optimizer,
        "--iterations", str(MAX_ITERATIONS),
        "--history-size", str(HISTORY_SIZE),
        "--window-days", str(WINDOW_DAYS),
        "--relative-gradient-tolerance", str(RELATIVE_GRADIENT_TOLERANCE),
        "--convergence-patience", str(CONVERGENCE_PATIENCE),
        "--minimum-iterations", str(MINIMUM_ITERATIONS),
        "--device-resident-state", resident,
        "--jit-cost-and-grad", jit,
        "--cost-and-grad-schedule", schedule,
    ], cwd=whirls_dir, env=env, text=True, capture_output=True)
    log = process.stdout + process.stderr
    if process.returncode:
        print("\n".join(log.splitlines()[-120:]))
        raise RuntimeError(f"Échec de {label} (code {process.returncode})")
    marker = "VARDYN_MINIMIZER_BENCHMARK_JSON="
    matches = [line for line in process.stdout.splitlines() if line.startswith(marker)]
    if not matches:
        raise RuntimeError(f"Résultat JSON absent pour {label}")
    result = json.loads(matches[-1][len(marker):])
    result["log_tail"] = "\n".join(log.splitlines()[-25:])
    return result

def save_results(results):
    results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

print(f"GPU physique : {GPU}; seuil relatif : {RELATIVE_GRADIENT_TOLERANCE}; patience : {CONVERGENCE_PATIENCE}")


GPU physique : 1; seuil relatif : 0.1; patience : 2


In [3]:
temporary_root = Path(tempfile.mkdtemp(prefix="vardyn-convergence-"))
baseline_checkout = temporary_root / "historical"
results = []
try:
    command("git", "worktree", "add", "--detach", baseline_checkout, baseline_sha, cwd=repo_root)
    print("Benchmark HISTORIQUE + SCIPY…")
    results.append(run_variant("historique_scipy", "scipy", baseline_checkout, temporary_root / "cache-historical", "off", "off", "python"))
    save_results(results)
    print("Benchmark SCAN + OPTAX DÉCOUPLÉ…")
    results.append(run_variant("scan_optax_decoupled", "optax-decoupled", repo_root, temporary_root / "cache-decoupled", "on", "on", "scan"))
    save_results(results)
finally:
    if baseline_checkout.exists():
        command("git", "worktree", "remove", "--force", baseline_checkout, cwd=repo_root, check=False)
    shutil.rmtree(temporary_root, ignore_errors=True)
print(f"Résultats enregistrés dans {results_path}")


Benchmark HISTORIQUE + SCIPY…


Benchmark SCAN + OPTAX DÉCOUPLÉ…


Résultats enregistrés dans /home/flo/VarDyn/mapping/examples/WHIRLS/benchmark_4dvar_convergence_results.json


## Résultats à précision comparable

In [4]:
if not results and results_path.exists():
    results = json.loads(results_path.read_text(encoding="utf-8"))
for item in results:
    item["final_cost"] = item["cost_history"][-1]
    item["cost_reduction_percent"] = 100 * (1 - item["final_cost"] / item["initial_cost"])
    item["end_to_end_seconds"] = item["setup_seconds"] + item["total_numerical_seconds"]
    item["gpu_peak_mib"] = item["memory_after_minimization"]["peak_bytes_in_use_mib"]
columns = ["label", "converged", "iterations_completed", "function_evaluations", "final_relative_gradient_norm", "final_cost", "cost_reduction_percent", "cost_compile_and_first_evaluation_seconds", "optimizer_init_seconds", "minimization_seconds", "total_numerical_seconds", "end_to_end_seconds", "gpu_peak_mib", "status"]
display(pd.DataFrame(results)[columns].set_index("label"))


,converged,iterations_completed,function_evaluations,final_relative_gradient_norm,final_cost,cost_reduction_percent,cost_compile_and_first_evaluation_seconds,optimizer_init_seconds,minimization_seconds,total_numerical_seconds,end_to_end_seconds,gpu_peak_mib,status
label,,,,,,,,,,,,,
historique_scipy,True,18,20,0.088757,628848.60107,81.692277,77.469588,0.000000,84.452767,161.922355,362.851193,3367.824463,relative gradient criterion reached
scan_optax_decoupled,True,17,19,0.067589,665054.62500,80.638210,71.321239,0.469217,48.872470,120.662926,323.639969,3863.983154,relative gradient criterion reached


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for item in results:
    iteration = np.arange(len(item["cost_history"]))
    relative_gradient = np.asarray(item["gradient_norm_history"]) / item["gradient_norm_history"][0]
    cold_offset = item["cost_compile_and_first_evaluation_seconds"] + item["optimizer_init_seconds"]
    axes[0].plot(iteration, item["cost_history"], marker="o", label=item["label"])
    axes[1].plot(iteration, relative_gradient, marker="o", label=item["label"])
    axes[2].plot(cold_offset + np.asarray(item["cumulative_seconds"]), item["cost_history"], marker="o", label=item["label"])
axes[0].set(title="Fonction coût", xlabel="Itération acceptée", ylabel="J")
axes[1].axhline(RELATIVE_GRADIENT_TOLERANCE, color="black", linestyle="--", label="seuil")
axes[1].set(title="Critère d’arrêt", xlabel="Itération acceptée", ylabel="||gₖ|| / ||g₀||", yscale="log")
axes[2].set(title="Coût contre temps numérique froid", xlabel="Secondes", ylabel="J")
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()
plt.show()


/tmp/ipykernel_2245018/4083610715.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
historical, decoupled = results
print(f"Accélération minimisation : {historical['minimization_seconds'] / decoupled['minimization_seconds']:.2f}×")
print(f"Accélération numérique froide : {historical['total_numerical_seconds'] / decoupled['total_numerical_seconds']:.2f}×")
print(f"Accélération bout en bout : {historical['end_to_end_seconds'] / decoupled['end_to_end_seconds']:.2f}×")
print(f"Écart relatif du coût final : {(decoupled['final_cost'] - historical['final_cost']) / historical['final_cost']:.3e}")


Accélération minimisation : 1.73×
Accélération numérique froide : 1.34×
Accélération bout en bout : 1.12×
Écart relatif du coût final : 5.758e-02


### Lecture

Le premier passage sous le seuil ne suffit pas : deux itérations consécutives sont exigées. Si `converged` est faux, la variante a atteint le plafond de 50 itérations ou rencontré un échec de recherche linéaire. La comparaison principale est le temps nécessaire pour satisfaire le même critère, pas le coût obtenu après un nombre arbitraire d’itérations.